# 5d Explore EMNIST Letters and Digits

Understand the difference between MNIST and NIST-derived EMNIST, fix orientation, and interpret label mappings.

**Budget:** 15–30 minutes, plus an optional large dataset download; a CPU and a few thousand images are enough. No Klique endpoint or token is needed for the default path.

**Before you start:** complete the [JupyterLab introduction](05_Running_Jupyter_Lab.ipynb), select a Python 3.10+ kernel, and use the dependency setup below if its packages are missing. Run every cell from a fresh kernel in order. The default is offline: cached real datasets are used when present; otherwise the helper explicitly identifies an 8×8 scikit-learn digits fallback. That fallback is **not MNIST, EMNIST, or Fashion-MNIST**, and its metrics are not comparable to theirs. Downloads are opt-in; read [dataset sources and terms](DATASETS.md) first.

Save your work in the approved persistent workspace. No live Klique launch or restore is established by these local exercises.


## Install the notebook dependencies if needed

The next cell installs the shared CPU packages into **this notebook's selected Python kernel** using `%pip`. It includes **scikit-learn**, whose import name is `sklearn`, plus NumPy, pandas, matplotlib, Pillow, and IPython.

If the packages are missing or this is a fresh environment, set `INSTALL_PACKAGES = True` and run the cell once. Only do this when installation is permitted and pip is configured to use your approved package source. If the environment is already prepared, leave it `False`. The cell finds `requirements.txt` whether the kernel starts in the repository root or `05_Jupyter`.

After a successful installation, change the flag back to `False`, use **Kernel → Restart Kernel**, and run the notebook from the top. A restart clears old imports and variables. If installation fails or the source is blocked, stop here and ask the administrator to prepare the environment or approved mirror; do not bypass TLS checks.

PyTorch/CUDA, interactive widgets, ClearML, and live gateway extras remain optional. See the [support guide](README.md#prepare-one-approved-python-environment) before enabling them. Package installation can use the network; the lesson's default dataset path remains offline.


In [ ]:
from pathlib import Path
import sys

# Change to True only when this kernel needs the section's packages installed.
INSTALL_PACKAGES = False

SETUP_HERE = Path.cwd().resolve()
SETUP_LAB = SETUP_HERE if (SETUP_HERE / "image_lab.py").is_file() else SETUP_HERE / "05_Jupyter"
REQUIREMENTS = SETUP_LAB / "requirements.txt"
if not REQUIREMENTS.is_file():
    raise FileNotFoundError("Start this kernel in the repository root or 05_Jupyter folder with requirements.txt present.")
print("Selected kernel Python:", sys.executable)

if INSTALL_PACKAGES:
    from IPython import get_ipython

    shell = get_ipython()
    if shell is None:
        raise RuntimeError("Run this setup cell in the notebook's IPython Python kernel.")
    # Equivalent to %pip install -r "{REQUIREMENTS}"; targets the selected kernel.
    shell.run_line_magic("pip", f'install -r "{REQUIREMENTS}"')
    print("Check pip output for errors. After success, set INSTALL_PACKAGES = False, restart the kernel, and run from the top.")
else:
    print("Installation skipped. If imports fail, set INSTALL_PACKAGES = True and rerun this cell first.")


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
LAB = HERE if (HERE / "image_lab.py").is_file() else HERE / "05_Jupyter"
if not (LAB / "image_lab.py").is_file():
    raise RuntimeError("Start this kernel in the repository root or 05_Jupyter folder.")
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))
from image_lab import load_image_data, train_baseline
SEED = 42
OUTPUT = LAB / "lab_outputs"
OUTPUT.mkdir(exist_ok=True)


## An explicit large-data choice

[EMNIST](https://www.nist.gov/itl/products-and-services/emnist-dataset) extends the MNIST-style format using NIST Special Database 19. `letters` has 26 classes with original labels 1–26; the helper makes those 0–25. `balanced` has 47 digit/letter classes and merges some visually similar case variants. It is not a simple A–Z/a–z/0–9 mapping. The helper corrects the stored orientation before display and training.

The official combined archive is about 560 MB compressed and expands much larger. Loading a small subset does not make the initial archive small. The large-download flag is separate; check disk space, deployment policy, and dataset terms first. The default run demonstrates the mapping/orientation contract with an explicitly synthetic asymmetric fixture. It does **not** train or claim performance on letters. See [sources, cache, and limits](DATASETS.md).


In [ ]:
# No network by default. See the support guide for the optional EMNIST dependency/cache.
EMNIST_SPLIT = "emnist_letters"  # or "emnist_balanced"
ALLOW_EMNIST_DOWNLOAD = False
emnist = None
try:
    emnist = load_image_data(EMNIST_SPLIT, allow_download=ALLOW_EMNIST_DOWNLOAD,
                            allow_emnist_download=ALLOW_EMNIST_DOWNLOAD,
                            cache_dir=OUTPUT / "data", limit_train=3000,
                            limit_test=600, seed=SEED)
except (FileNotFoundError, RuntimeError, ImportError) as exc:
    print("Real EMNIST unavailable:", str(exc))
    print("Running only the synthetic orientation/label demonstration below.")
if emnist is not None:
    print(emnist.name, emnist.provenance)
    print("Class labels:", emnist.class_names)


In [ ]:
# Synthetic L-shaped fixture, not a handwriting dataset.
upright = np.zeros((8, 8), dtype=np.float32)
upright[1:7, 1:3] = 1; upright[5:7, 1:6] = 1
stored_fixture = upright.T
corrected = stored_fixture.T
np.testing.assert_array_equal(corrected, upright)
original_letter_labels = np.array([1, 2, 26])
zero_based = original_letter_labels - 1
assert zero_based.tolist() == [0, 1, 25]
print("Letters mapping example:", list(zip(original_letter_labels, zero_based, ["A", "B", "Z"])))
fig, axes = plt.subplots(1, 2, figsize=(5, 2))
for ax, image, title in zip(axes, [stored_fixture, corrected], ["Stored fixture", "Corrected fixture"]):
    ax.imshow(image, cmap="gray", vmin=0, vmax=1); ax.set_title(title); ax.axis("off")
plt.show()


In [ ]:
if emnist is not None:
    fig, axes = plt.subplots(2, 6, figsize=(10, 4))
    for index, ax in enumerate(axes.flat):
        ax.imshow(emnist.train_images[index], cmap="gray", vmin=0, vmax=1)
        ax.set_title(emnist.class_names[int(emnist.train_labels[index])]); ax.axis("off")
    plt.show()
    emnist_model, emnist_metrics = train_baseline(emnist, C=1.0, max_iter=100, seed=SEED)
    print("Real EMNIST held-out result:", emnist_metrics)
else:
    print("No EMNIST model trained. No letter/digit accuracy is available.")


## Checkpoint, exercise, and cleanup

**Checkpoint:** You distinguish the synthetic offline contract check from a real EMNIST run, and know why letters and balanced labels cannot be interchanged.

**Try it:** With approved real data, compare letters and balanced class names. Inspect both orientation and label mapping before training; explain why O/0 and I/1 can be ambiguous.

**Troubleshooting:** if imports fail, select the kernel where the section requirements were installed. If a data host is blocked, keep downloads off and use the named offline fixture, or ask your administrator to stage the official data in the cache. Do not disable TLS verification. A convergence warning means the solver hit its iteration budget; record it rather than silently claiming an optimum.

Save the notebook and useful artifacts under `lab_outputs/`; it is ignored by Git but may still enter workspace snapshots. Clear outputs before sharing if you used private data. Shut down unused kernels, then stop the managed session and verify its stopped state as in the introduction. Closing a browser tab is not cleanup.

**Next:** [5e: Compare Experiments](05e_Compare_Experiments.ipynb) · **Index:** [Jupyter labs](README.md) · [Orientation](../00_Orientation/00_Orientation.ipynb)
